# libKriging vs STK (Octave)

This notebook compares **mlibkriging** (libKriging's Octave/MATLAB binding)
and **STK** (https://github.com/stk-kriging/stk, "a Small Toolbox for
Kriging") along two axes:

1. **Mimicry**: reproduce with `mlibkriging` a "default" Kriging fit
   equivalent to STK's `stk_model(@stk_materncov52_aniso, d)` +
   `stk_param_estim` (Matern 5/2 anisotropic covariance, constant/ordinary
   trend, hyperparameters by maximum likelihood), and check that both give
   near-identical predictions. This follows the same protocol as
   `bench/comparison/run_stk.m`, which benchmarks STK against libKriging's
   other bindings on shared datasets.
2. **Package-specific feature**: STK's `stk_generate_samplepaths(model, xi,
   zi, xt, n)` produces *conditional* sample paths of the fitted Gaussian
   process in a single call (`stk_example_kb08` in STK's own examples) --
   the same idea as `Kriging.simulate()` on the `mlibkriging` side, already
   used in this series' `libKriging_vs_OpenTURNS` notebook for pylibkriging
   vs `GaussianProcessConditionalCovariance`. Both draw spatially-coherent
   posterior sample paths (not independent per-point noise), which is what
   a downstream Monte Carlo analysis needs from a Kriging metamodel.

Steps:
1. Check that `mlibkriging` is built and STK is on the Octave path.
2. Load both packages.
3. Define the Branin test function and a Latin Hypercube design.
4. Mimicry: fit both packages' default Kriging on Branin 2D and overlay
   predictions on 1D slices.
5. Special feature: conditional sample paths, `simulate()` vs
   `stk_generate_samplepaths()`.
6. Argument correspondence table.

## 0. Setup

Requires `mlibkriging` (built from this repository, see
`bindings/Octave/README.md`) and STK
(`git clone --depth 1 https://github.com/stk-kriging/stk.git`), both on the
Octave path -- the same layout used by `bench/comparison/run_stk.m` in CI.
Set the environment variable `STK_PATH` to STK's clone directory if it is
not a sibling of the libKriging repository.

In [ ]:
repo_root = fullfile(pwd(), "..", "..");
build_path = fullfile(repo_root, "build", "installed", "bindings", "Octave");
if ~exist(fullfile(build_path, ["mLibKriging." mexext]), "file")
  error(["mlibkriging not found at " build_path ...
         " -- please build first (see bindings/Octave/README.md)."]);
endif

stk_path = getenv("STK_PATH");
if isempty(stk_path)
  stk_path = fullfile(repo_root, "..", "stk");
endif
if ~exist(fullfile(stk_path, "stk_init.m"), "file")
  error(["STK not found at " stk_path ...
         " -- clone https://github.com/stk-kriging/stk and set STK_PATH."]);
endif

printf("mlibkriging build: %s\n", build_path);
printf("STK path         : %s\n", stk_path);

## 1. Load both packages

In [ ]:
addpath(build_path);
addpath(stk_path);
stk_init;

printf("mlibkriging loaded\n");
printf("STK loaded\n");

## 2. Branin function and design of experiments

Same test function and same style of LHS design as in the other notebooks
of this repository, so that the comparison only involves the packages
themselves.

In [ ]:
function z = branin(X)
  x1 = X(:,1) * 15 - 5;
  x2 = X(:,2) * 15;
  z = (x2 - 5/(4*pi^2) * x1.^2 + 5/pi * x1 - 6).^2 ...
      + 10 * (1 - 1/(8*pi)) * cos(x1) + 10;
endfunction

rand("seed", 42);
n = 20; d = 2;
X = zeros(n, d);
for j = 1:d
  perm = randperm(n);
  X(:,j) = (perm' - rand(n,1)) / n;
endfor
y = branin(X);

rand("seed", 7);
Xtest = rand(300, d);

### Cross-section helper

Rather than comparing side-by-side heatmaps, we overlay both models' mean
prediction (+/- 1 standard deviation ribbon) on the same axes along 1D
slices of the input space, at a few fixed values of $x_2$.

In [ ]:
function plot_slices(preds, names, colors, slice_vals)
  x1_seq = linspace(0, 1, 200)';
  figure("position", [0, 0, 1400, 350]);
  for k = 1:numel(slice_vals)
    s = slice_vals(k);
    Xs = [x1_seq, s * ones(numel(x1_seq), 1)];
    y_true = branin(Xs);
    subplot(1, numel(slice_vals), k);
    plot(x1_seq, y_true, "k-", "linewidth", 2); hold on;
    for i = 1:numel(preds)
      [m, sdv] = preds{i}(Xs);
      plot(x1_seq, m, "color", colors{i}, "linewidth", 1.5);
      fill([x1_seq; flipud(x1_seq)], [m + sdv; flipud(m - sdv)], colors{i}, ...
           "facealpha", 0.15, "edgecolor", "none");
    endfor
    title(sprintf("x2 = %.2f", s));
    xlabel("x1"); ylabel("y");
    if k == 1
      legend(["true", names], "location", "best");
    endif
    hold off;
  endfor
endfunction

## 3. Mimicry: `Kriging()` vs `stk_model` + `stk_param_estim`

Common settings: Matern 5/2 anisotropic covariance (one length-scale per
input), constant/ordinary trend, no nugget beyond STK's own numerical
safeguards, hyperparameters by maximum likelihood. `stk_param_estim` uses
STK's own data-driven initial guess (no explicit multistart control, unlike
`mlibkriging`'s `optim = "BFGS10"`), so -- as with the DiceKriging/RobustGaSP
comparisons -- we rely on `BFGS10`'s 10 restarts to land on the same global
optimum that STK's heuristic initial guess is designed to find directly.

In [ ]:
k_lk = Kriging(y, X, "matern5_2", "constant", false, "BFGS10", "LL");
printf("mlibkriging theta: %s  sigma2: %.4f\n", mat2str(k_lk.theta(), 4), k_lk.sigma2());

model = stk_model(@stk_materncov52_aniso, d);
model = stk_param_estim(model, X, y);
% STK stores hyperparameters in its own log-parameterisation (see
% "help stk_materncov52_aniso"), not directly comparable to mlibkriging's
% theta/sigma2 -- so the two fits are compared through their predictions
% below instead.

In [ ]:
[mean_lk, sd_lk] = k_lk.predict(Xtest, true, false, false);

zp = stk_predict(model, X, y, Xtest);
mean_stk = zp.mean;
sd_stk = sqrt(max(zp.var, 0));

printf("Max |mean_lk - mean_stk| : %.6f\n", max(abs(mean_lk - mean_stk)));
printf("Correlation              : %.6f\n", corr(mean_lk, mean_stk));

function [m, s] = lk_predict(k, Xs)
  [m, s] = k.predict(Xs, true, false, false);
endfunction

function [m, s] = stk_predict_wrap(mdl, xi, zi, Xs)
  zp = stk_predict(mdl, xi, zi, Xs);
  m = zp.mean;
  s = sqrt(max(zp.var, 0));
endfunction

preds_mimicry = {@(Xs) lk_predict(k_lk, Xs), @(Xs) stk_predict_wrap(model, X, y, Xs)};
plot_slices(preds_mimicry, {"mlibkriging::Kriging", "STK::stk_predict"}, ...
            {[0.12 0.47 0.71], [0.89 0.10 0.11]}, [0.25 0.5 0.75]);

Both fits reproduce the same overall predictor: `mlibkriging`'s
`Kriging(kernel="matern5_2", regmodel="constant", objective="LL")` closely
matches STK's `stk_model(@stk_materncov52_aniso, d)` fitted with
`stk_param_estim` -- the mean curves and uncertainty ribbons overlap almost
everywhere, and the predicted means over 300 random test points are highly
correlated, mirroring the agreement already seen with DiceKriging,
RobustGaSP and the Python packages in this series.

## 4. Package-specific feature: conditional sample paths

STK exposes `stk_generate_samplepaths(model, xi, zi, xt, n)` as a one-call
shortcut for drawing `n` *conditional* sample paths of the fitted process at
test points `xt`, given observations `(xi, zi)` -- internally equivalent to
generating unconditional paths with `stk_generate_samplepaths(model, xt,
n)`, computing kriging weights with `stk_predict`, and combining them with
`stk_conditioning` (STK's own `stk_example_kb08` demonstrates both routes
side by side). `mlibkriging`'s `Kriging.simulate(nsim, seed, X, will_update)`
plays the same role, already compared to OpenTURNS'
`GaussianProcessConditionalCovariance` in this series' `libKriging_vs_OpenTURNS`
notebook.

In [ ]:
n_paths = 6;
x1_seq = linspace(0, 1, 150)';
Xs = [x1_seq, 0.5 * ones(numel(x1_seq), 1)];

sims_lk = k_lk.simulate(n_paths, 123, Xs, false);       % n x n_paths
sims_stk = stk_generate_samplepaths(model, X, y, Xs, n_paths);

idx = find(abs(X(:,2) - 0.5) < 0.15);

figure("position", [0, 0, 1400, 500]);

subplot(1, 2, 1);
plot(x1_seq, branin(Xs), "k-", "linewidth", 2); hold on;
for i = 1:n_paths
  plot(x1_seq, sims_lk(:,i), "color", [0.12 0.47 0.71], "linewidth", 1);
endfor
plot(X(idx,1), y(idx), "ko", "markerfacecolor", "k", "markersize", 6);
title("mlibkriging: Kriging.simulate()");
xlabel("x1"); ylabel("y"); hold off;

subplot(1, 2, 2);
plot(x1_seq, branin(Xs), "k-", "linewidth", 2); hold on;
for i = 1:n_paths
  plot(x1_seq, sims_stk(:,i), "color", [0.89 0.10 0.11], "linewidth", 1);
endfor
plot(X(idx,1), y(idx), "ko", "markerfacecolor", "k", "markersize", 6);
title("STK: stk_generate_samplepaths()");
xlabel("x1"); ylabel("y"); hold off;

Both packages produce smooth, spatially-coherent sample paths (not
independent per-point noise) that pinch towards near-zero variance at/near
the design points and fan out with the same Matern 5/2 roughness in
between -- a direct visual check that `mlibkriging`'s `simulate()` and
STK's `stk_generate_samplepaths()` implement the same conditional-simulation
idea, just as already observed for OpenTURNS'
`GaussianProcessConditionalCovariance` elsewhere in this series.

## 5. Argument correspondence table

| STK | mlibkriging equivalent | Notes |
|---|---|---|
| `stk_model(@stk_materncov52_aniso, d)` | `kernel = "matern5_2"` | STK also has `stk_materncov32_aniso`, `stk_materncov_iso`, `stk_gausscov_aniso`, `stk_expcov_aniso`, etc.; `_iso` variants -> a single shared length-scale |
| default trend (constant, "ordinary kriging") | `regmodel = "constant"` | STK supports richer trends via `model.lm` |
| `stk_param_estim(model, x, z)` | `optim = "BFGS10", objective = "LL"` | STK's default initial guess is data-driven (similar spirit to `DiceKriging::km()`'s heuristic); no explicit multistart knob like `mlibkriging`'s `BFGSk` |
| `stk_predict(model, xi, zi, xt)` | `predict(Xs, true, false, false)` | STK returns a dataframe with `.mean`/`.var` fields; `mlibkriging` returns `[mean, stdev]` |
| `stk_generate_samplepaths(model, xi, zi, xt, n)` | `simulate(nsim, seed, X, will_update)` | STK's one-call conditional-simulation shortcut vs. the explicit `stk_generate_samplepaths` + `stk_predict` + `stk_conditioning` route (`stk_example_kb05`/`stk_example_kb08`) |
| `model.lognoisevariance` | `noise = <vector>` (heterogeneous noise model) | not exercised in this notebook |